In [ ]:
# 1. Google Drive 연동
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/datasets'
# train_dir = f'{base_dir}/train'
# test_dir = f'{base_dir}/test'
# val_dir = f'{base_dir}/val'

In [ ]:
import os
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from torch.utils.data.dataloader import default_collate
import cv2
from tqdm import tqdm

In [ ]:
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
        print(f"Extracted {zip_file_path} to {extract_dir}")


base_dir = f"{extract_dir}/datasets"
train_dir = f"{base_dir}/train"
val_dir = f"{base_dir}/val"
test_dir = f"{base_dir}/test"

# 데이터 경로 확인
print("Train Directory:", train_dir)
print("Test Directory:", test_dir)
print("Validation Directory:", val_dir)

In [ ]:
# 2. 데이터셋 로드

# Numpy 배열을 PyTorch 텐서로 변환하는 함수 정의//
class NumpyToTensor:
    def __call__(self, array):
        if isinstance(array, np.ndarray):
            if array.ndim == 3:  # 이미지의 경우 (높이, 너비, 채널)
                array = np.transpose(array, (2, 0, 1))  # (높이, 너비, 채널) -> (채널, 높이, 너비)
                array = array / 255.0
            elif array.ndim == 2:  # 마스크의 경우 (높이, 너비)
                array = np.expand_dims(array, axis=0)  # (1, 높이, 너비)로 변경 (채널 추가)
                array = array / 255.0  # 0과 1 사이로 정규화
            return torch.from_numpy(array).float()
        return array

class NpyDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = sorted(os.listdir(os.path.join(root_dir, 'original_images')))
        self.weld_mask_paths = sorted(os.listdir(os.path.join(root_dir, 'weld_masks')))
        self.unpretreated_mask_paths = sorted(os.listdir(os.path.join(root_dir, 'unpretreated_masks')))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = os.path.join(self.root_dir, 'original_images', self.image_paths[idx])
        weld_mask_path = os.path.join(self.root_dir, 'weld_masks', self.weld_mask_paths[idx])
        unpretreated_mask_path = os.path.join(self.root_dir, 'unpretreated_masks', self.unpretreated_mask_paths[idx])

        try:
            # 이미지 로드 및 검증
            image = np.load(image_path)
            weld_mask = np.load(weld_mask_path)
            unpretreated_mask = np.load(unpretreated_mask_path)

            # 예상되는 크기를 검증 (예: 이미지 크기는 768x768x3, 마스크는 768x768)
            if image.shape != (768, 768, 3):
                print(f"Invalid image shape: {image.shape} at {image_path}, skipping this file.")
                return None

            if weld_mask.shape != (768, 768) or unpretreated_mask.shape != (768, 768):
                print(f"Invalid mask shape: weld={weld_mask.shape}, unpretreated={unpretreated_mask.shape} at {image_path}, skipping this file.")
                return None

        except Exception as e:
            print(f"Error loading file {image_path} or corresponding masks: {e}")
            return None

        if self.transform:
            image = self.transform(image)
            weld_mask = self.transform(weld_mask)
            unpretreated_mask = self.transform(unpretreated_mask)

        masks = np.stack([weld_mask, unpretreated_mask], axis=0)

        return image, masks

# Transform 정의 (이미지 크기를 256x256으로 리사이즈하고 Numpy 배열을 Tensor로 변환)
transform = transforms.Compose([
    NumpyToTensor(),  # Numpy 배열을 Tensor로 변환
])

train_dataset = NpyDataset(train_dir, transform=transform)
val_dataset = NpyDataset(val_dir, transform=transform)
test_dataset = NpyDataset(test_dir, transform=transform)

# None 값을 필터링하는 collate_fn 정의
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))  # None 값을 필터링
    if len(batch) == 0:  # 배치에 데이터가 없는 경우 None 반환
        return None
    return default_collate(batch)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)


In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


In [ ]:
# 3. Attention_UNet++ 모델 정의

class AttentionUNet(nn.Module):
    def __init__(self):
        super(AttentionUNet, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )

        def up_conv(in_channels, out_channels):
            return nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)

        self.encoder1 = conv_block(3, 16)
        self.encoder2 = conv_block(16, 32)
        self.encoder3 = conv_block(32, 64)
        self.encoder4 = conv_block(64, 128)
        self.encoder5 = conv_block(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.upconv4 = up_conv(256, 128)
        self.decoder4 = conv_block(256, 128)
        self.upconv3 = up_conv(128, 64)
        self.decoder3 = conv_block(128, 64)
        self.upconv2 = up_conv(64, 32)
        self.decoder2 = conv_block(64, 32)
        self.upconv1 = up_conv(32, 16)
        self.decoder1 = conv_block(32, 16)

        self.attention4 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.attention3 = AttentionGate(F_g=64, F_l=64, F_int=32)
        self.attention2 = AttentionGate(F_g=32, F_l=32, F_int=16)
        self.attention1 = AttentionGate(F_g=16, F_l=16, F_int=8)

        self.final_conv = nn.Conv2d(16, 2, kernel_size=1)

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        enc4 = self.encoder4(self.pool(enc3))
        enc5 = self.encoder5(self.pool(enc4))

        dec4 = self.upconv4(enc5)
        att4 = self.attention4(dec4, enc4)
        dec4 = torch.cat((dec4, att4), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        att3 = self.attention3(dec3, enc3)
        dec3 = torch.cat((dec3, att3), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        att2 = self.attention2(dec2, enc2)
        dec2 = torch.cat((dec2, att2), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        att1 = self.attention1(dec1, enc1)
        dec1 = torch.cat((dec1, att1), dim=1)
        dec1 = self.decoder1(dec1)

        return self.final_conv(dec1)


In [ ]:
# 4. 모델 학습 및 평가

# 모델, 옵티마이저, 손실 함수 정의
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttentionUNet().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
# criterion = nn.BCEWithLogitsLoss()

train_losses = []
val_losses = []

# DiceLoss 정의
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, outputs, targets):
        smooth = 1.0  # 안정성을 위한 작은 값
        outputs = torch.sigmoid(outputs)  # 확률값으로 변환
        outputs_flat = outputs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (outputs_flat * targets_flat).sum()
        dice = (2.0 * intersection + smooth) / (outputs_flat.sum() + targets_flat.sum() + smooth)
        return 1 - dice  # Dice 계수 -> 손실로 변환

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

# 조합된 손실 함수 정의
def combined_loss(outputs, targets, alpha=0.2, beta=0.8):
    bce = bce_loss(outputs, targets)
    dice = dice_loss(outputs, targets)
    return alpha * bce + beta * dice

####################################################################################
# 구글 드라이브에 모델 가중치를 저장할 경로 설정
model_save_path = '/content/drive/MyDrive/UNet_model/Attention_unet_Augmentation_Dice80_{epoch}.pth'
####################################################################################


# start_epoch = 12

# # 저장된 모델 가중치를 불러오기
# saved_model_path = f'/content/drive/MyDrive/UNet_model/Attention_unet_Dice_model_channelDown_epoch_{start_epoch}.pth'
# model.load_state_dict(torch.load(saved_model_path))
# print(f"Model weights loaded from {saved_model_path}")

# 학습 루프
epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0

     # tqdm을 사용하여 학습 진행률 표시
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=True)

    for images, masks in train_loader:
        if images is None or masks is None:  # None 값이 있는 경우 건너뜀
            continue
        images, masks = images.to(device), masks.to(device)

        # masks 텐서에서 불필요한 차원 제거
        masks = masks.squeeze(2)  # 차원이 [batch_size, 2, height, width] 형식으로 변환

        optimizer.zero_grad()
        outputs = model(images)

        # loss = criterion(outputs, masks)
        loss = combined_loss(outputs, masks, alpha=0.2, beta=0.8)  # 조합된 손실 함수 사용
        loss.backward()

        # # BCEWithLogitsLoss 사용
        # loss = bce_loss(outputs, masks)
        # loss.backward()

        # Gradient Clipping 추가
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)


        optimizer.step()

        train_loss += loss.item()


        # 실시간으로 배치당 손실을 출력
        # print(f"Loss: {loss.item():.4f}")

        # progress_bar.set_postfix({'Loss': loss.item()})
        progress_bar.update(1)

    # 에포크가 끝난 후, 손실 출력
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    progress_bar.set_postfix({'Epoch Train Loss': train_loss})
    progress_bar.close()


####################################################################################
    # 매 에포크가 끝난 후, Train Loss 계산 후 가중치 저장
    torch.save(model.state_dict(), model_save_path.format(epoch=epoch+1))
    print(f"Model saved for epoch {epoch+1} at {model_save_path.format(epoch=epoch+1)}")
####################################################################################


###################################
    val_loss = 0
    model.eval()

    # tqdm을 사용하여 검증 진행률 표시
    progress_bar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=True)

    with torch.no_grad():
        for images, masks in val_loader:
            if images is None or masks is None:  # None 값이 있는 경우 건너뜀
                continue
            images, masks = images.to(device), masks.to(device)

            # masks 텐서에서 불필요한 차원 제거
            masks = masks.squeeze(2)  # 차원이 [batch_size, 2, height, width] 형식으로 변환

            outputs = model(images)
            # loss = criterion(outputs, masks)
            loss = combined_loss(outputs, masks, alpha=0.5, beta=0.5)
            loss = bce_loss(outputs, masks)
            val_loss += loss.item()

            progress_bar_val.update(1)

    # 에포크가 끝난 후, 검증 손실 출력
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    progress_bar_val.set_postfix({'Epoch Val Loss': val_loss})
    progress_bar_val.close()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
###################################

# Train Losses와 Val Losses 시각화
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, epochs+1), val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Validation Loss over Epochs')
plt.legend()
plt.show()

In [ ]:
# #두 마스크(용접선, 전처리 안 한 면)를 비교하여 각각의 정확도를 계산하는 함수

def calculate_accuracy(prediction, ground_truth):
    # Sigmoid로 예측값을 0과 1 사이 값으로 변환
    prediction = torch.sigmoid(prediction).cpu().detach().numpy()

    # Ground Truth의 불필요한 차원을 제거 (squeeze 적용)
    ground_truth = ground_truth.cpu().numpy().squeeze()  # 배치 차원 제거

    # 각 마스크에 대해 0.5 기준으로 이진화
    weld_mask_pred = prediction[0] > 0.5  # 용접선 예측 마스크
    unpretreated_mask_pred = prediction[1] > 0.5  # 전처리 안 한 면 예측 마스크

    # Ground Truth 마스크
    weld_mask_gt = ground_truth[0] > 0.5  # 용접선 실제 마스크
    unpretreated_mask_gt = ground_truth[1] > 0.5  # 전처리 안 한 면 실제 마스크

    # # 중앙 5x5 픽셀 부분을 추출하여 출력
    # center_row = weld_mask_pred.shape[0] // 2
    # center_col = weld_mask_pred.shape[1] // 2

    # print("Weld Mask Prediction (Center 5x5):")
    # print(weld_mask_pred[center_row-2:center_row+3, center_col-2:center_col+3])

    # print("Unpretreated Mask Prediction (Center 5x5):")
    # print(unpretreated_mask_pred[center_row-2:center_row+3, center_col-2:center_col+3])

    # # 2차원 흑백 마스크 이미지 시각화
    # fig, ax = plt.subplots(2, 2, figsize=(10, 10))

    # ax[0, 0].imshow(weld_mask_pred, cmap='gray')  # 용접선 예측 마스크 시각화
    # ax[0, 0].set_title('Weld Mask Prediction')

    # ax[0, 1].imshow(weld_mask_gt, cmap='gray')  # 용접선 Ground Truth 마스크 시각화
    # ax[0, 1].set_title('Weld Mask Ground Truth')

    # ax[1, 0].imshow(unpretreated_mask_pred, cmap='gray')  # 전처리 안 한 면 예측 마스크 시각화
    # ax[1, 0].set_title('Unpretreated Mask Prediction')

    # ax[1, 1].imshow(unpretreated_mask_gt, cmap='gray')  # 전처리 안 한 면 Ground Truth 마스크 시각화
    # ax[1, 1].set_title('Unpretreated Mask Ground Truth')

    plt.show()

    # IoU 기반 정확도 계산: 교집합 / 합집합
    weld_intersection = (weld_mask_pred & weld_mask_gt).sum()
    weld_union = (weld_mask_pred | weld_mask_gt).sum()
    weld_iou_accuracy = weld_intersection / weld_union if weld_union > 0 else 0

    unpretreated_intersection = (unpretreated_mask_pred & unpretreated_mask_gt).sum()
    unpretreated_union = (unpretreated_mask_pred | unpretreated_mask_gt).sum()

    # Ground Truth가 비어 있으면 정확도 100%로 처리
    if unpretreated_mask_gt.sum() == 0:
        unpretreated_iou_accuracy = weld_iou_accuracy
    else:
        unpretreated_iou_accuracy = unpretreated_intersection / unpretreated_union if unpretreated_union > 0 else 0

    return weld_iou_accuracy, unpretreated_iou_accuracy

In [ ]:
# 5. 시각화 함수: 예측 결과와 Ground Truth 시각화


def visualize_prediction(image, original_prediction, ground_truth_masks=None, alpha=0.6):
    # Sigmoid를 통해 예측 값을 0과 1 사이 값으로 변환
    prediction = torch.sigmoid(original_prediction)
    prediction = prediction.cpu().detach().numpy()

    # 예측된 마스크: 용접선(파란색), 전처리 안 한 면(빨간색), 교집합(초록색)
    weld_mask_pred = prediction[0] > 0.5  # 용접선 클래스
    unpretreated_mask_pred = prediction[1] > 0.5  # 전처리 안 한 면 클래스
    intersection_pred = weld_mask_pred & unpretreated_mask_pred  # 교집합 클래스

    # 원본 이미지를 RGB로 변환 (배경은 원본 이미지 그대로 사용)
    image_np = np.transpose(image.cpu().numpy(), (1, 2, 0))  # (C, H, W) -> (H, W, C)
    image_np = (image_np * 255).astype(np.uint8)  # 원본 이미지 값을 uint8로 변환

    # 결과 이미지를 생성
    result_image = np.copy(image_np)
    result_image[weld_mask_pred] = [0, 0, 255]  # 파란색 (용접선)
    result_image[unpretreated_mask_pred] = [255, 0, 0]  # 빨간색 (전처리 안 한 면)
    result_image[intersection_pred] = [0, 255, 0]  # 초록색 (교집합)

    # 알파 블렌딩: 원본 이미지와 예측 마스크 결합
    blended_image = cv2.addWeighted(image_np, 1 - alpha, result_image.astype(np.uint8), alpha, 0)

     # "Prediction with Alpha Blending" 이미지를 Google Drive에 저장
    # if save_index is not None:
    #     save_path = f'/content/drive/MyDrive/Prediction_image/prediction_with_alpha_blending_{save_index}.png'
    #     cv2.imwrite(save_path, cv2.cvtColor(blended_image, cv2.COLOR_RGB2BGR))  # RGB -> BGR 변환하여 저장


    # 시각화
    fig, ax = plt.subplots(1, 3, figsize=(20, 5))

    ax[0].imshow(image_np)  # 원본 이미지
    ax[0].set_title("Original Image")

    ax[1].imshow(blended_image)  # 예측된 마스크와 원본 이미지 알파 블렌딩
    ax[1].set_title("Prediction with Alpha Blending")

    if ground_truth_masks is not None:
        # Ground Truth 마스크 (실제 마스크): 용접선과 전처리 안 한 면
        ground_truth = ground_truth_masks.cpu().numpy().squeeze()  # 배치 차원 제거
        weld_mask_gt = ground_truth[0] > 0.5  # 2D로 변환 (용접선)
        unpretreated_mask_gt = ground_truth[1] > 0.5  # 2D로 변환 (전처리 안 한 면)
        intersection_gt = weld_mask_gt & unpretreated_mask_gt  # 교집합

        # RGB로 변환된 Ground Truth 이미지 생성
        ground_truth_image = np.copy(image_np)  # 원본 이미지 복사
        ground_truth_image[weld_mask_gt] = [0, 0, 255]  # 파란색 (용접선)
        ground_truth_image[unpretreated_mask_gt] = [255, 0, 0]  # 빨간색 (전처리 안 한 면)
        ground_truth_image[intersection_gt] = [0, 255, 0]  # 초록색 (교집합)

        ax[2].imshow(ground_truth_image)  # Ground Truth 마스크 시각화
        ax[2].set_title("Ground Truth Mask")

        # 정확도 계산 추가
        weld_iou_acc, unpretreated_iou_acc = calculate_accuracy(original_prediction, ground_truth_masks)

        # 정확도 출력
        print(f'Weld Mask Accuracy: {weld_iou_acc * 100:.2f}%')
        print(f'Unpretreated Mask Accuracy: {unpretreated_iou_acc * 100:.2f}%')

        average_accuracy = (weld_iou_acc + unpretreated_iou_acc) / 2
        print(f'Average Accuracy: {average_accuracy * 100:.2f}%')



    plt.show()

In [ ]:
# 6. 테스트 데이터로 결과 시각화
model.eval()

weld_accuracies = []
unpretreated_accuracies = []

with torch.no_grad():
    for i, (images, masks) in enumerate(test_loader):
        images = images.to(device)
        masks = masks.to(device)
        outputs = model(images)

        for j in range(len(images)):
            visualize_prediction(images[j], outputs[j], masks[j])

            # 각 이미지의 정확도 계산
            weld_acc, unpretreated_acc = calculate_accuracy(outputs[j], masks[j])
            weld_accuracies.append(weld_acc)
            unpretreated_accuracies.append(unpretreated_acc)

        if i == 10:  # 배치 10개 시각화
            break

# 전체 평균 정확도 계산
mean_weld_accuracy = sum(weld_accuracies) / len(weld_accuracies)
mean_unpretreated_accuracy = sum(unpretreated_accuracies) / len(unpretreated_accuracies)

# 용접선과 전처리 안 한 면 전체 평균 정확도 계산
overall_accuracy = (sum(weld_accuracies) + sum(unpretreated_accuracies)) / (len(weld_accuracies) + len(unpretreated_accuracies))


print(f"평균 용접선 정확도: {mean_weld_accuracy:.4f}")
print(f"평균 전처리 안 한 면 정확도: {mean_unpretreated_accuracy:.4f}")
print(f"전체 평균 정확도: {overall_accuracy:.4f}")